# IEP-1001: 단순 청킹(Simple Chunking) Context Recall 평가

**목적**: Chunk Size / Overlap 조합 6가지를 비교하여 최적 청킹 구간 탐색  
**평가 지표**: Context Recall (RAGAS)  
**Judge LLM**: llama3.1:8b  
**임베딩 모델**: jhgan/ko-sroberta-multitask  
**대상 문서**: IPCC AR6 SYR 한글 번역본 (188페이지)

| Case | Chunk Size | Overlap |
|------|-----------|---------|
| CASE 1 | 500 | 100 |
| CASE 2 | 800 | 150 |
| CASE 3 | 1000 | 200 |
| CASE 4 | 1200 | 240 |
| CASE 5 | 1500 | 300 |
| CASE 6 | 2000 | 400 |

> **실행 환경**: Google Colab T4 GPU  
> **사전 준비**: Google Drive에 `IPCC_data/KO_IPCC_AR6_SYR_FullVolume.pdf` 및 `IPCC_data/golen_dataset.csv` 업로드 필요


## 1. 환경 설정

### 1-1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1-2. 시스템 패키지 및 Ollama 설치

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

### 1-3. Python 패키지 설치

In [ ]:
# 문서 파싱
!pip install -qU pymupdf

# LangChain 핵심 패키지
!pip install -qU langchain-community langchain-huggingface langchain-ollama

# 벡터 DB
!pip install -qU chromadb

# 평가 프레임워크
!pip install -qU ragas

### 1-4. Ollama 서버 시작

In [ ]:
import subprocess
import time

process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("Ollama 서버 시작 중...")
time.sleep(10)
print("Ollama 서버 실행 완료")

### 1-5. Ollama 모델 다운로드

In [ ]:
!ollama pull llama3.1:8b    # Judge LLM
!ollama pull nomic-embed-text  # 임베딩 (RAGAS 내부 평가용)

## 2. RAG 파이프라인 설정

### 2-1. 청킹 설정

실험할 Chunk Size / Overlap 조합 6가지를 정의한다.

In [ ]:
# 실험 대상 청킹 케이스 정의
CHUNK_CASES = [
    {"name": "CASE1", "chunk_size": 500,  "chunk_overlap": 100},
    {"name": "CASE2", "chunk_size": 800,  "chunk_overlap": 150},
    {"name": "CASE3", "chunk_size": 1000, "chunk_overlap": 200},
    {"name": "CASE4", "chunk_size": 1200, "chunk_overlap": 240},
    {"name": "CASE5", "chunk_size": 1500, "chunk_overlap": 300},
    {"name": "CASE6", "chunk_size": 2000, "chunk_overlap": 400},
]

# 실행할 케이스 선택 (전체 실행 시 CHUNK_CASES 그대로 사용)
TARGET_CASE = CHUNK_CASES[0]  # 원하는 케이스로 변경

print(f"실행 케이스: {TARGET_CASE['name']} "
      f"(chunk_size={TARGET_CASE['chunk_size']}, "
      f"chunk_overlap={TARGET_CASE['chunk_overlap']})")

### 2-2. 문서 로드

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

FILE_PATH = "/content/drive/MyDrive/IPCC_data/KO_IPCC_AR6_SYR_FullVolume.pdf"

loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()

print(f"문서 로드 완료: {len(docs)}페이지")

### 2-3. 청킹

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=TARGET_CASE["chunk_size"],
    chunk_overlap=TARGET_CASE["chunk_overlap"]
)

chunked_docs = text_splitter.split_documents(docs)

print(f"청킹 완료: {len(chunked_docs)}개 청크 생성")
print(f"평균 청크 크기: {sum(len(d.page_content) for d in chunked_docs) // len(chunked_docs)}자")

### 2-4. 임베딩 모델 설정

한국어 특화 모델 `jhgan/ko-sroberta-multitask` 사용.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

print("임베딩 모델 로드 완료: jhgan/ko-sroberta-multitask")

### 2-5. 벡터 DB 저장 (ChromaDB)

In [ ]:
from langchain_community.vectorstores import Chroma

CHROMA_DIR = f"./chroma_db_{TARGET_CASE['name']}"

vectorstore = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    persist_directory=CHROMA_DIR
)

print(f"벡터 DB 저장 완료: {CHROMA_DIR}")

### 2-6. 검색 테스트

In [ ]:
# 검색 동작 확인용 스모크 테스트
SMOKE_QUERY = "지구 온도가 2도 상승하면?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
smoke_results = retriever.invoke(SMOKE_QUERY)

print(f"쿼리: {SMOKE_QUERY}\n")
for i, doc in enumerate(smoke_results):
    print(f"--- 검색 결과 {i+1} (p.{doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:200] + "...\n")

## 3. RAGAS Context Recall 평가

### 3-1. 평가 지표 설정

In [ ]:
from ragas.metrics import ContextRecall

metrics = [ContextRecall()]

print("평가 지표 설정 완료: ContextRecall")

### 3-2. Judge LLM 및 임베딩 설정

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Judge LLM
langchain_llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0,
    num_predict=1024,
    top_p=0.1
)
ragas_llm = LangchainLLMWrapper(langchain_llm)
print("Judge LLM 설정 완료: llama3.1:8b")

# 임베딩
langchain_embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)
ragas_embeddings = LangchainEmbeddingsWrapper(langchain_embeddings)
print("임베딩 설정 완료: nomic-embed-text")

### 3-3. 골든 데이터셋 로드

In [ ]:
import pandas as pd

GOLDEN_PATH = "/content/drive/MyDrive/IPCC_data/golen_dataset.csv"
golden_dataset = pd.read_csv(GOLDEN_PATH).to_dict('records')

print(f"골든 데이터셋 로드 완료: {len(golden_dataset)}개")
pd.DataFrame(golden_dataset).head(3)

### 3-4. 검색 결과 추가 (Retrieved Contexts 구성)

In [ ]:
all_dataset = []

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

for gold in golden_dataset:
    search_results = retriever.invoke(gold["user_input"])
    retrieved_contexts = [doc.page_content for doc in search_results]

    all_dataset.append({
        "user_input": gold["user_input"],
        "retrieved_contexts": retrieved_contexts,
        "reference": gold["reference"],
    })

print(f"검색 결과 구성 완료: {len(all_dataset)}개")

### 3-5. 평가 데이터셋 구성

In [ ]:
from ragas import EvaluationDataset

eval_dataset = EvaluationDataset.from_pandas(pd.DataFrame(all_dataset))

print(f"평가 데이터셋 구성 완료: {len(eval_dataset)}개")

### 3-6. Context Recall 평가 실행

In [ ]:
from ragas import evaluate
from ragas.run_config import RunConfig

my_run_config = RunConfig(
    max_workers=1,
    timeout=600,
    max_retries=10,
)

results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=my_run_config,
    batch_size=1,
    raise_exceptions=False
)

result_df = results.to_pandas()

# 결과 저장
OUTPUT_PATH = f"/content/drive/MyDrive/IPCC_data/recall_{TARGET_CASE['name']}.csv"
result_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print(f"평가 완료 — 결과 저장: {OUTPUT_PATH}")
display(result_df)

## 4. 결과 요약

In [ ]:
# 유형별 평균 Recall 및 NaN 현황
summary = result_df.groupby("user_input").agg(
    context_recall=("context_recall", "mean")
).reset_index()

nan_count = result_df["context_recall"].isna().sum()
mean_recall = result_df["context_recall"].mean()

print(f"=== {TARGET_CASE['name']} 결과 요약 ===")
print(f"청크 크기: {TARGET_CASE['chunk_size']} / 오버랩: {TARGET_CASE['chunk_overlap']}")
print(f"전체 평균 Context Recall : {mean_recall:.4f}")
print(f"NaN 발생 수              : {nan_count}개")
display(result_df[["user_input", "context_recall"]].describe())